In [2]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [3]:
loader = PyPDFLoader("ugrulebook.pdf")
documents = loader.load()
print(len(documents))

44


In [4]:
for doc in documents[1:5]:
    print(doc)

page_content='2                                      
Move to Index 
Rules are classified into three separate categories as follows: (I) those which may be implemented 
within a department by DUGC/DPGC, (ii) those that require a decision at the level of Associate/ Dean 
Academic Progamme or UGAPEC/PGAPEC, based on recommendations from the department bodies 
(iii) those that need to be discussed in the Senate for a decision. 
 
Therefore, rules are colored with one of three colors. 
1. The color green indicates that the final authority for rule is the Convener DUGC 
2. The color yellow, and underlined means that the final authority is Associate Dean (Academic 
Programme)/ Dean (Academic Programme) 
3. The color yellow, without an underline means that the Convener, UGAPEC is the authority. 
4. The color blue means that the final authority is the Senate 
5. The rule which is uncolored, is to be implemented strictly' metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word

In [17]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=0,
    length_function=len,
    is_separator_regex=False
)

In [18]:
naive_chunks = text_splitter.split_documents(documents)
for chunk in naive_chunks[10:15]:
  print(chunk.page_content+ "\n")

1. The color green indicates that the final authority for rule is the Convener DUGC

2. The color yellow, and underlined means that the final authority is Associate Dean (Academic

Programme)/ Dean (Academic Programme)

3. The color yellow, without an underline means that the Convener, UGAPEC is the authority.

4. The color blue means that the final authority is the Senate



In [19]:
type(naive_chunks)

list

In [20]:
type(semantic_chunks)

list

In [6]:
pip install sentence-transformers --quiet

Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain.embeddings import HuggingFaceEmbeddings

embed_model = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")

C:\Users\Dell\AppData\Local\Temp\ipykernel_22396\1250464492.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embed_model = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
c:\Python 3.10\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from langchain_experimental.text_splitter import SemanticChunker
# from langchain_openai.embeddings import OpenAIEmbeddings

semantic_chunker = SemanticChunker(embed_model, breakpoint_threshold_type="percentile")

In [7]:
semantic_chunks = semantic_chunker.create_documents([d.page_content for d in documents])

c:\Python 3.10\lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


In [8]:
print(len(semantic_chunks))

115


In [9]:
for semantic_chunk in semantic_chunks[10:15]:
    print(semantic_chunk.page_content + "\n")

Programmes. IIT Bombay gives freedom to its various academic units to tailor their aca demic programmes as per 
their specific needs, within the frame work of the Rules and Regulations approved by the Senate 
from time to time. Keeping in view the new technical developments and to allow students some 
freedom to explore topics of their interest, IIT Bombay revised its Undergraduate Programmes cur-
riculum (w.e.f. the Academic Year 2013). The revised curriculum, provides additional opportunities 
and flexibility for students to optimize their learning experience. This needs continuous and meticu-
lous planning of the academic profile on the part of each student to fully utilize the opportunities. The students, and parents/ guardians, are therefore, advised in their own interest to get fully familiar 
with the Academic system of the Institute. Student’s attention is brought particularly to the assess-
ment procedures and the specific rules governing the grading system, academic performan

In [21]:
from langchain_chroma import Chroma
persist_directory = 'docs1'

vectordb = Chroma.from_documents(
    documents=semantic_chunks,
    embedding=embed_model,
    persist_directory=persist_directory
)


: 

In [ ]:
from langchain_google_genai import GoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain.vectorstores import Chroma
import os

In [ ]:
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

template = """
You are an intelligent Question-Answer bot. Your task is to respond to questions by using the given context. Each answer should follow the structured format below, including the answer and the source of information.

### Response Format:
1. **Answer**: Provide a clear and complete answer to the question based only on the provided context.
2. **Source**: Explicitly mention the source of the answer in parentheses at the end. For example, 'Source: Section 2.5.1, Minor Program.'

If the answer is not in the context, say: "I don’t know based on the provided information." Do not attempt to make up an answer or provide false information. Use only the context to respond.

### Example:
**Answer**: IIT Bombay follows a 10-point grading system where the grades range from AP (Exceptional Performance) to FR (Fail).  
**Source**: Section 6.5, Grading System.

Now, use the context below to answer the question in the format provided.

Context:
{context}

Question: {input}
Answer with Source:
"""

# Now you can use this template with ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", template),
        ("human", "{input}")
    ]
)

questions = [
    "What is the grading system at IIT Bombay?",
    "What does the 'AP' grade signify at IIT Bombay?",
    "What is the significance of the 'PP' and 'NP' grades?",
    "Is attendance mandatory for all courses?",
    "What are the core academic phases of IIT Bombay's undergraduate programs?"
]

# Reference answers (corresponding to each question)
reference_answers = [
    "IIT Bombay follows a 10-point grading system where the grades range from 'AP' (Exceptional Performance) to 'FR' (Fail). Source: Grading System section.",
    "The 'AP' grade signifies exceptional performance and is awarded only in courses where the number of registered students is more than 50. Source: Grading System section.",
    "The 'PP' (Pass) and 'NP' (Not Pass) grades are used for non-credit courses such as NCC, NSO, and NSS. Source: Grading System section.",
    "Yes, IIT Bombay expects 100percent attendance from its students. A 'DX' grade is given if attendance falls below 80%. Source: Attendance section.",
    "The undergraduate program consists of three phases: intense study of sciences and humanities, study of engineering sciences, and specialized subjects in chosen areas. Source: Introduction section."
]

In [ ]:
llm = GoogleGenerativeAI(model="gemini-2.0-flash")# gemini-1.5-pro-latest
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(vectordb.as_retriever(), question_answer_chain)

for i, question in enumerate(questions):
    # Invoke rag_chain for the question
    result = rag_chain.invoke({"input": question})
    
    # Print the question and both responses
    print(f"Question {i+1}: {question}")
    print(f"Chatbot Answer: {result['answer']}")
    print(f"Your Answer: {reference_answers[i]}")
    
    # Output separator
    print("\n" + "-"*50 + "\n")